In [2]:
import os, torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image

CONFIG = {
    "img_size": (224, 224),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

t = transforms.Compose([
    transforms.Resize(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print(f"✅ تم تجهيز البيئة على: {CONFIG['device']}")

✅ تم تجهيز البيئة على: cpu


In [3]:
class SiameseTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        eff = models.efficientnet_b0(weights=None)
        self.backbone = eff.features
        self.feature_dim = 1280
        self.pos_embedding = nn.Parameter(torch.randn(1, 49, self.feature_dim))
        layer = nn.TransformerEncoderLayer(d_model=self.feature_dim, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        
        # التعديل هنا: زودنا طبقة Dropout عشان تطابق الموديل المتسيف
        self.fc = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2), # دي الطبقة اللي كانت ناقصة ومسببة المشكلة
            nn.Linear(256, 128)
        )

    def forward_one(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), self.feature_dim, -1).permute(0, 2, 1)
        x = self.transformer(x + self.pos_embedding)
        return self.fc(torch.mean(x, dim=1))

    def forward(self, i1, i2): return self.forward_one(i1), self.forward_one(i2)

model = SiameseTransformer().to(CONFIG['device'])
print("✅ تم تعديل الهيكل ليطابق الأوزان المتسيفة.")

✅ تم تعديل الهيكل ليطابق الأوزان المتسيفة.


In [25]:
model_path = "A:\\Gproject\\handwriting-recognition\\src\\app\\api\\Modeltraning\\final_signature_model.pth"
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=CONFIG['device']))
    model.eval()
    print("✅ تم تحميل الأوزان بنجاح! الموديل جاهز.")
else:
    print("❌ المسار غلط! اضغط على اسم الداتا سيت على اليمين وانسخ مسار الملف (Copy Path) وحطه مكان model_path.")

✅ تم تحميل الأوزان بنجاح! الموديل جاهز.


In [36]:
def final_test(img1_path, img2_path):
    model.eval()
    im1 = t(Image.open(img1_path).convert("RGB")).unsqueeze(0).to(CONFIG['device'])
    im2 = t(Image.open(img2_path).convert("RGB")).unsqueeze(0).to(CONFIG['device'])
    
    with torch.no_grad():
        o1, o2 = model(im1, im2)
        # الحساب الدقيق للتشابه
        sim = F.cosine_similarity(o1, o2).item()
    
    # تحويل من (1 to -1) لنسبة مئوية (0 to 100)
    display_prob = (sim + 1) / 2 * 100
    
    print(f"\n{'='*30}")
    print(f"🔍 تفاصيل الفحص الفني:")
    print(f"📏 قيمة الـ Cosine Similarity: {sim:.4f}")
    print(f"📊 نسبة التطابق المحسوبة: {display_prob:.2f}%")
    
    # القرار بناءً على قيمة التشابه الخام
    if sim > 0.4: # عتبة منطقية للـ Transformer
        print(f"القرار النهائي: ✅ توقيع أصلي (Genuine)")
    else:
        print(f"القرار النهائي: ❌ توقيع مزور (Forged)")
    print(f"{'='*30}")

# جرب الاختبار تاني
orig = "A:\\Gproject\\handwriting-recognition\\src\\app\\api\\Modeltraning\\Images\\O11.png"
test = "A:\\Gproject\\handwriting-recognition\\src\\app\\api\\Modeltraning\\Images\\O1.png" 

final_test(orig, test)


🔍 تفاصيل الفحص الفني:
📏 قيمة الـ Cosine Similarity: 0.9999
📊 نسبة التطابق المحسوبة: 99.99%
القرار النهائي: ✅ توقيع أصلي (Genuine)
